# Lesson 05: Prompt Engineering Fundamentals

In this notebook, you will run all 5 prompt engineering exercises using `tiktoken` and `google-genai`:
1. Tokenize the Jupiter astronomy passage.
2. Basic text completion (`"oh say can you see"`).
3. Test fabrication on `"Martian War of 2076"` with and without an explicit "out".
4. Summarize the Jupiter passage for a 2nd-grade student.
5. Run a multi-turn conversation with a sarcastic `system_instruction` persona.

In [ ]:
!pip install -q -U google-genai tiktoken

In [ ]:
import os
import tiktoken
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
jupiter = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar System. "
    "It is a gas giant with a mass more than two and a half times that of all the other planets "
    "in the Solar System combined, while being less than one-thousandth the mass of the Sun."
)

enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(jupiter)
print(f"tiktoken count: {len(tokens)} | First 8 token bytes: {[enc.decode_single_token_bytes(t) for t in tokens[:8]]}")

In [ ]:
if client:
    # Exercise 4: Instruction + Primary Content
    r1 = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f'Summarize this for a second-grade student:\n"""{jupiter}"""',
    )
    print("2nd-Grade Summary:\n", r1.text)

    # Exercise 5: Multi-turn with system_instruction
    chat = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(system_instruction="You are a witty, slightly sarcastic sports trivia assistant."),
    )
    print("\nTurn 1:", chat.send_message("Who won the World Series in 2020?").text)
    print("Turn 2:", chat.send_message("Where was it played?").text)
else:
    print("[Offline Demo Mode] 2nd-Grade Summary: Jupiter is the biggest planet in our solar system, made mostly of gas!")
    print("[Offline Demo Mode] Turn 1: Oh, only the Los Angeles Dodgers in 2020—played at Globe Life Field in Arlington, Texas.")